# 02_bronze_EU · da landing para a Bronze

**Lakehouse Sísmico Europeu · aula 2 de 4**

| | |
|---|---|
| **Entra** | os arquivos `.json` de `/Volumes/sismos_eu/bronze/landing/terremotos/` |
| **Sai** | `sismos_eu.bronze.terremotos_raw` · 1 linha = 1 resposta da API |
| **Memória** | `/Volumes/sismos_eu/bronze/checkpoints/terremotos_raw` |
| **Regra** | rodar de novo lê só arquivo novo: a 2ª execução grava **zero linhas** |

> Dados: EMSC-CSEM SeismicPortal, https://www.seismicportal.eu. Os termos do portal pedem crédito à fonte ao usar os dados.

A Bronze guarda a resposta **como veio**: nenhuma coluna renomeada, nenhum tipo convertido, nenhum
evento filtrado. Só ganha duas colunas de auditoria: quando entrou e de qual arquivo veio.

Isso vale também para o que é estranho nos dados do EMSC: `time` fica como **texto ISO**, `evtype` fica como
**código** e a profundidade negativa em `coordinates` continua negativa. Corrigir é trabalho da Silver.

## 0 · Garantir a estrutura
O notebook não confia que alguém criou o catálogo antes: ele garante sozinho. Com `IF NOT EXISTS`, rodar de
novo não muda nada. É o que deixa o notebook funcionar num workspace novo, ou como primeira tarefa de um Job.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS sismos_eu;
CREATE SCHEMA  IF NOT EXISTS sismos_eu.bronze;
CREATE VOLUME  IF NOT EXISTS sismos_eu.bronze.landing;
CREATE VOLUME  IF NOT EXISTS sismos_eu.bronze.checkpoints;

In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/sismos_eu/bronze/landing/terremotos/"
CKPT    = "/Volumes/sismos_eu/bronze/checkpoints/terremotos_raw"
TABELA  = "sismos_eu.bronze.terremotos_raw"

# o * casa o escopo (pasta escopo=XXXXXXXX), que depende do filtro usado no 01_extract_EU
UM_ARQUIVO = f"{LANDING}escopo=*/ano=2025/mes=07.json"

### Recomeçar do zero (só quando você pedir)
O widget `recomecar` aparece no topo do notebook e vem sempre em **`nao`**. Em `sim`, a célula abaixo apaga a tabela **e** o checkpoint, juntos.
Apagar só um dos dois deixa o par inconsistente:
- só a tabela: o checkpoint acha que já leu tudo, e nada volta para a tabela;
- só o checkpoint: tudo é relido e a tabela fica com linhas repetidas.

Use quando os arquivos da landing foram **reescritos** com outro conteúdo (o Auto Loader reconhece arquivo pelo caminho e não os relê).
**Volte o widget para `nao` antes de rodar de novo**, senão cada execução apaga tudo outra vez.
O manifesto do `01_extract_EU` não entra nessa limpeza: ele é o diário da extração e não depende do Auto Loader.

In [0]:
dbutils.widgets.dropdown("recomecar", "nao", ["nao", "sim"], "Recomeçar do zero?")

if dbutils.widgets.get("recomecar") == "sim":
    print("recomecar = sim: apagando a tabela e o checkpoint, juntos")
    spark.sql(f"DROP TABLE IF EXISTS {TABELA}")
    dbutils.fs.rm(CKPT, recurse=True)
else:
    print("recomecar = nao: nada é apagado")

recomecar = nao: nada é apagado


## 1 · Por que `multiLine`
Por padrão, o Spark lê JSON como **um registro por linha** (o formato *JSON Lines*). A resposta do EMSC
é **um** objeto só, com todos os eventos dentro do array `features`, e o arquivo vem **formatado em várias linhas**:
no arquivo de julho de 2025, 623 eventos ocuparam 16.199 linhas, cerca de 26 por evento. Lido do jeito padrão,
cada linha vira um JSON incompleto. Veja:

In [0]:
print("linhas de texto no arquivo:", spark.read.text(UM_ARQUIVO).count())

padrao = spark.read.json(UM_ARQUIVO)
print("leitura padrão, colunas:", padrao.columns)

linhas de texto no arquivo: 16199
leitura padrão, colunas: ['_corrupt_record']


Uma coluna só, `_corrupt_record`: o objeto não foi entendido. (O `count()` dessa leitura não serve de medida, por isso não o usamos aqui:
o que prova o problema é a lista de colunas.) Agora com `multiLine`:

In [0]:
certo = spark.read.option("multiLine", "true").json(UM_ARQUIVO)
print("com multiLine")
print("  colunas:  ", certo.columns)
print("  registros:", certo.count())
display(certo.select("type", "metadata.count", F.size("features").alias("eventos")))

com multiLine
  colunas:   ['features', 'metadata', 'type']
  registros: 1


type,count,eventos
FeatureCollection,623,623


**1 arquivo = 1 linha**, com os eventos todos dentro do array `features`. É exatamente esse o grão da Bronze.
Explodir o array em uma linha por evento é trabalho da Silver, na aula 3.

## 2 · Auto Loader
O `spark.read` de cima lê **tudo, toda vez**. O Auto Loader (`format("cloudFiles")`) lê **só o que ainda não leu**,
e anota o que já leu no checkpoint.

| Opção | Para quê |
|---|---|
| `cloudFiles.format = json` | o formato dos arquivos |
| `multiLine = true` | 1 arquivo = 1 objeto |
| `cloudFiles.inferColumnTypes = true` | sem ela, o Auto Loader grava todo campo de JSON como **texto**, inclusive o array `features` inteiro |
| `cloudFiles.schemaLocation` | onde o schema inferido fica guardado, para a próxima execução não adivinhar de novo |
| `cloudFiles.partitionColumns = escopo,ano` | transforma as pastas `escopo=XXXXXXXX` e `ano=2025` em colunas |
| `checkpointLocation` | a memória: quais arquivos já foram gravados na tabela |
| `trigger(availableNow=True)` | processa o que existe agora e **para**. É streaming que se comporta como batch |

Uma opção que não aparece na célula, mas age sozinha: `cloudFiles.schemaEvolutionMode`, que no modo padrão é `addNewColumns`.
Se um dia o EMSC acrescentar um campo novo à resposta, a execução **falha uma vez**, grava o schema novo em `schemaLocation`
e a seguinte passa. No Job da aula 4, o retry cobre isso.

In [0]:
antes = spark.table(TABELA).count() if spark.catalog.tableExists(TABELA) else 0

consulta = (
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiLine", "true")                       # 1 arquivo = 1 objeto
        .option("cloudFiles.inferColumnTypes", "true")     # array continua array, número continua número
        .option("cloudFiles.schemaLocation", f"{CKPT}/_schema")
        .option("cloudFiles.partitionColumns", "escopo,ano")   # pastas viram colunas
        .load(LANDING)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
        .option("checkpointLocation", CKPT)
        .trigger(availableNow=True)                        # processa o que há e para
        .toTable(TABELA)
)
consulta.awaitTermination()

In [0]:
r = (spark.table(TABELA)
        .agg(F.count("*").alias("linhas"), F.sum(F.size("features")).alias("eventos"))
        .first())

print(TABELA)
print(f"  nesta execução:     {r.linhas - antes} linhas")
print(f"  total na tabela:    {r.linhas} linhas")
print(f"  eventos (features): {r.eventos:,}".replace(",", "."))

sismos_eu.bronze.terremotos_raw
  nesta execução:     0 linhas
  total na tabela:    12 linhas
  eventos (features): 13.252


## 3 · O momento da aula
Volte à célula do Auto Loader e rode **de novo**, junto com a de cima. Esperado:
```
  nesta execução:     0 linhas
  total na tabela:    12 linhas
```
O checkpoint lembra quais arquivos já foram lidos. É idempotência **sem apagar nada**, e é o que vai deixar o
Job da aula 4 tentar de novo sozinho, de madrugada.

## 4 · O que o Delta registrou
A versão 0 é o `CREATE TABLE`. Depois vêm as versões `STREAMING UPDATE` da primeira execução: o Auto Loader
pode dividir os 12 arquivos em um ou mais lotes, e cada lote vira uma versão. Some a coluna `linhas_gravadas`: dá 12.
Uma execução sem arquivo novo **não deve criar versão**, porque não há o que gravar. Confira na tabela abaixo: depois
da segunda execução, a última versão continua a mesma.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {TABELA}")
        .select("version", "timestamp", "operation",
                F.col("operationMetrics")["numOutputRows"].alias("linhas_gravadas")))

version,timestamp,operation,linhas_gravadas
2,2026-11-22T21:15:27.818Z,STREAMING UPDATE,2
1,2026-11-22T21:15:26.749Z,STREAMING UPDATE,10
0,2026-11-22T21:15:21.298Z,CREATE TABLE,null


## 5 · Conferências
No EMSC, o `metadata.count` conta os eventos **que vieram na resposta**, e não o total que a API tinha. Sozinho, ele só mostra
que o cabeçalho do arquivo concorda com o array dentro dele. Por isso a conferência abaixo traz três números por arquivo:
- `count_da_api`: o que o `/count` disse na hora da extração, lido do **manifesto** gravado pelo `01_extract_EU`;
- `diz_o_arquivo`: o `metadata.count` do próprio arquivo;
- `na_bronze`: quantos eventos a Bronze guardou de fato.

`bate = true` quando os três concordam. Um `bate = NULL` significa arquivo sem linha no manifesto. E um `bate = false` depois de
rodar o `01` de novo é um sinal útil: o arquivo foi reescrito, a API deu outro número, e a Bronze **ficou para trás** (veja a seção 7).

In [0]:
%sql
WITH bronze AS (
  SELECT regexp_extract(_source_file, '(escopo=.*)$', 1) AS arquivo,
         metadata.count                                  AS diz_o_arquivo,
         size(features)                                  AS na_bronze
  FROM sismos_eu.bronze.terremotos_raw
),
manifesto AS (
  SELECT arquivo, esperado
  FROM (
    SELECT *, row_number() OVER (PARTITION BY arquivo ORDER BY extraido_em DESC) AS rn
    FROM sismos_eu.bronze.extracoes_manifesto
  )
  WHERE rn = 1
)
SELECT b.arquivo,
       m.esperado      AS count_da_api,
       b.diz_o_arquivo,
       b.na_bronze,
       (m.esperado = b.na_bronze AND b.diz_o_arquivo = b.na_bronze) AS bate
FROM bronze b
LEFT JOIN manifesto m ON b.arquivo = m.arquivo
ORDER BY b.arquivo

arquivo,count_da_api,diz_o_arquivo,na_bronze,bate
escopo=9c121421/ano=2025/mes=01.json,757,823,823,false
escopo=9c121421/ano=2025/mes=02.json,3064,3095,3095,false
escopo=9c121421/ano=2025/mes=03.json,789,917,917,false
escopo=9c121421/ano=2025/mes=04.json,974,1010,1010,false
escopo=9c121421/ano=2025/mes=05.json,768,804,804,false
escopo=9c121421/ano=2025/mes=06.json,672,690,690,false
escopo=9c121421/ano=2025/mes=07.json,623,642,642,false
escopo=9c121421/ano=2025/mes=08.json,1462,1488,1488,false
escopo=9c121421/ano=2025/mes=09.json,846,879,879,false
escopo=9c121421/ano=2025/mes=10.json,1067,1116,1116,false


### Unicidade: cada evento aparece uma vez?
O `bate` prova **completude** (a API disse N, chegaram N, a Bronze guardou N). Não prova unicidade: o `/count` conta as linhas que a API devolve, repetidas ou não.
Foi assim que a primeira extração passou despercebida com **461 eventos repetidos** (13.252 linhas para 12.791 ids): o `endtime` do EMSC, quando recebe só a data,
cobria o dia inteiro, e o dia 1º de cada mês entrava em dois arquivos.

As duas consultas abaixo mostram, por arquivo, quantos eventos e quantos ids distintos há, e o primeiro e o último horário de cada mês.
Esperado depois da correção da janela: `eventos = ids_no_arquivo`, e os horários dentro do mês do arquivo.

In [0]:
%sql
SELECT regexp_extract(_source_file, '(escopo=.*)$', 1)                    AS arquivo,
       size(features)                                                     AS eventos,
       size(array_distinct(transform(features, x -> x.properties.unid)))  AS ids_no_arquivo,
       array_min(transform(features, x -> x.properties.time))             AS primeiro_evento,
       array_max(transform(features, x -> x.properties.time))             AS ultimo_evento
FROM sismos_eu.bronze.terremotos_raw
ORDER BY arquivo

arquivo,eventos,ids_no_arquivo,primeiro_evento,ultimo_evento
escopo=9c121421/ano=2025/mes=01.json,823,823,2025-01-01T00:17:15.15Z,2025-02-01T23:44:16.0Z
escopo=9c121421/ano=2025/mes=02.json,3095,3095,2025-02-01T00:41:43.99Z,2025-03-01T23:17:15.99Z
escopo=9c121421/ano=2025/mes=03.json,917,917,2025-03-01T00:41:48.41Z,2025-04-01T22:29:42.42Z
escopo=9c121421/ano=2025/mes=04.json,1010,1010,2025-04-01T01:30:04.74Z,2025-05-01T23:47:25.59Z
escopo=9c121421/ano=2025/mes=05.json,804,804,2025-05-01T00:33:17.09Z,2025-06-01T21:38:46.7Z
escopo=9c121421/ano=2025/mes=06.json,690,690,2025-06-01T03:10:10.67Z,2025-07-01T23:01:08.65Z
escopo=9c121421/ano=2025/mes=07.json,642,642,2025-07-01T00:31:40.96Z,2025-08-01T18:20:59.46Z
escopo=9c121421/ano=2025/mes=08.json,1488,1488,2025-08-01T01:33:19.06Z,2025-09-01T23:06:35.25Z
escopo=9c121421/ano=2025/mes=09.json,879,879,2025-09-01T01:26:23.63Z,2025-10-01T23:49:45.13Z
escopo=9c121421/ano=2025/mes=10.json,1116,1116,2025-10-01T01:17:27.49Z,2025-11-01T23:37:09.89Z


In [0]:
%sql
-- entre arquivos: repetidos = 0 quando cada evento aparece em um único mês
SELECT count(*)                                                AS eventos,
       count(DISTINCT f.properties.unid)                       AS ids_distintos,
       count(*) - count(DISTINCT f.properties.unid)            AS repetidos
FROM (SELECT explode(features) AS f FROM sismos_eu.bronze.terremotos_raw)

eventos,ids_distintos,repetidos
13252,12791,461


E o schema. Repare em quatro coisas:
- `features` é `ARRAY<STRUCT<...>>`, igual à API, com os tipos que o próprio JSON tem: `mag`, `lat`, `lon` e `depth` são `DOUBLE`. Isso é ler o JSON direito, e não converter.
  Sem o `inferColumnTypes`, `features` inteiro viria como uma `STRING` gigante.
- `time` e `lastupdate` são `STRING` (texto ISO 8601). Virar timestamp é trabalho da Silver.
- `escopo` e `ano` vieram do nome das pastas.
- `_rescued_data` foi criada sozinha: é para onde vai um campo que não cabe no schema, em vez de ser perdido.

In [0]:
%sql
DESCRIBE sismos_eu.bronze.terremotos_raw

col_name,data_type,comment
features,"array,type:string>,id:string,properties:struct,type:string>>",null
metadata,struct,null
type,string,null
escopo,string,null
ano,int,null
_rescued_data,string,null
_ingested_at,timestamp,null
_source_file,string,null
# Clustering Information,,
# col_name,data_type,comment


## 6 · Primeira olhada nos eventos
Ainda sem Silver: `explode` na hora da consulta, só para conhecer o dado. Cinco descobertas que viram decisão na aula 3.

In [0]:
%sql
-- evtype é um código: veja quais existem antes de decidir o que é terremoto
SELECT f.properties.evtype AS tipo, count(*) AS eventos
FROM (SELECT explode(features) AS f FROM sismos_eu.bronze.terremotos_raw)
GROUP BY ALL
ORDER BY eventos DESC

tipo,eventos
ke,13204
se,23
ue,15
fe,8
si,2


In [0]:
%sql
-- quem mede: cada instituto (auth) tem seu jeito de calcular a magnitude (magtype)
SELECT f.properties.auth AS instituto, f.properties.magtype AS tipo_magnitude, count(*) AS eventos
FROM (SELECT explode(features) AS f FROM sismos_eu.bronze.terremotos_raw)
GROUP BY ALL
ORDER BY eventos DESC
LIMIT 15

instituto,tipo_magnitude,eventos
EMSC,ml,8539
THE,ml,1306
NOA,ml,457
IMO,m,416
EMSC,mb,366
AFAD,ml,312
INGV,ml,258
PDA,ml,184
KOERI,ml,179
EMSC,mw,174


In [0]:
%sql
-- os maiores do ano: time ainda é texto, o to_timestamp só traduz na consulta
SELECT f.properties.unid                          AS id,
       f.properties.flynn_region                  AS regiao,
       f.properties.mag                           AS magnitude,
       f.properties.magtype                       AS tipo_magnitude,
       to_timestamp(f.properties.time)            AS horario_utc,
       f.properties.auth                          AS instituto
FROM (SELECT explode(features) AS f FROM sismos_eu.bronze.terremotos_raw)
ORDER BY magnitude DESC
LIMIT 5

id,regiao,magnitude,tipo_magnitude,horario_utc,instituto
20250310_0000037,JAN MAYEN ISLAND REGION,6.5,mw,2025-03-10T02:33:13.990Z,EMSC
20250522_0000028,"CRETE, GREECE",6.2,mw,2025-05-22T03:19:34.600Z,EMSC
20250423_0000104,WESTERN TURKEY,6.2,mw,2025-04-23T09:49:11.930Z,EMSC
20250810_0000200,WESTERN TURKEY,6.1,mw,2025-08-10T16:53:47.890Z,EMSC
20251027_0000264,WESTERN TURKEY,6.0,mw,2025-10-27T19:48:28.840Z,EMSC


In [0]:
%sql
-- a profundidade: em coordinates vem negativa, em properties.depth vem positiva
SELECT f.properties.depth       AS depth_properties,
       f.geometry.coordinates[2] AS depth_coordinates
FROM (SELECT explode(features) AS f FROM sismos_eu.bronze.terremotos_raw)
LIMIT 5

depth_properties,depth_coordinates
11.0,-11.0
5.6,-5.6
5.0,-5.0
7.0,-7.0
6.0,-6.0


In [0]:
%sql
-- fevereiro tem várias vezes mais eventos que os outros meses: de onde vêm?
SELECT f.properties.flynn_region AS regiao, count(*) AS eventos
FROM (SELECT explode(features) AS f FROM sismos_eu.bronze.terremotos_raw)
WHERE month(to_timestamp(f.properties.time)) = 2
GROUP BY ALL
ORDER BY eventos DESC
LIMIT 5

regiao,eventos
"DODECANESE ISLANDS, GREECE",2615
AEGEAN SEA,48
SOUTHERN ITALY,45
SOUTHERN GREECE,40
GREECE,36


### O que as consultas mostraram (primeira extração de 2025, **antes** de corrigir a janela)
As contagens abaixo são em **linhas**, e incluem os repetidos da seção 5. Depois de reextrair e recarregar, rode as consultas de novo: os números mudam um pouco.
- **`evtype`:** 13.204 eventos `ke`, 23 `se`, 15 `ue`, 8 `fe` e 2 `si`. Pela tabela de tipos do ISC, `ke` é *known earthquake*, `se` é *suspected earthquake*,
  `fe` é *felt earthquake* e `si` é *suspected induced event*. O `ue` não tem definição oficial que eu tenha confirmado: pela convenção da FDSN, a primeira letra
  indica a certeza (`u` é *unknown*). A Bronze guarda todos; o que conta como terremoto é decisão da Silver.
- **`auth` e `magtype`:** o `ml` (magnitude local) domina, com vários institutos (EMSC, THE, NOA, AFAD, INGV...), e cada um calcula o seu. O `mw` é raro, e os 5 maiores eventos do ano são todos `mw`.
  Magnitudes de tipos diferentes não são comparáveis.
- **Profundidade:** `coordinates[2]` é a mesma profundidade de `properties.depth` com o sinal trocado.
- **Fevereiro:** uma região, `DODECANESE ISLANDS, GREECE`, concentrava a grande maioria dos eventos do mês (2.615 contra 3.095 linhas no arquivo de fevereiro).
  **Esses números estavam inflados:** o dia 1º de fevereiro aparecia em dois arquivos, e o arquivo de fevereiro provavelmente trazia também o dia 1º de março.
  A concentração numa região é compatível com o enxame sísmico de Santorini–Amorgos, que o EMSC descreve em relatório especial como iniciado em 27 de janeiro de 2025,
  mas isso é uma hipótese (o nome da região, sozinho, não prova a localização) e o percentual exato só vale depois da correção.

## 7 · A resposta da pergunta do `01_extract_EU`
Rode o `01_extract_EU` de novo e depois este notebook: **0 linhas novas**, mesmo que o EMSC tenha revisado
eventos e os arquivos tenham mudado.

O Auto Loader reconhece arquivo pelo **caminho**. Arquivo reescrito com o mesmo nome não é relido
(`cloudFiles.allowOverwrites` é `false` por padrão). É ótimo para não duplicar, e ruim para quem quer a revisão.

Mas agora a Bronze **não fica cega**: rode a conferência da seção 5 de novo. Se o EMSC revisou o mês, o manifesto novo traz um
`/count` diferente do que a Bronze guardou, e `bate` aparece `false`. A desatualização deixa de ser silenciosa.

Na aula 4, a carga incremental resolve isso do jeito certo: cada execução grava um arquivo **novo**, só com o
que mudou desde a última. A API do EMSC aceita `updatedafter`, mas a documentação não diz o formato esperado:
isso será testado antes de montar o Job.